In [1]:
import sys
import os
import torch

In [2]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment

2025-02-22 16:23:59,734 - INFO - NumExpr defaulting to 8 threads.


In [8]:
from torch_geometric.data import Data
def print_subgraph_stats(data: Data, name: str = "Subgraph"):
    """
    Prints basic statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object
        name: Name/identifier for the subgraph (default="Subgraph")
    """
    print(f"{name} stats:")
    print(f"Number of nodes: {data.num_nodes}")
    print(f"Number of edges: {data.edge_index.size(1)}")
    print(f"Number of features: {data.x.size(1)}")
    print(f"Number of classes: {len(torch.unique(data.y))}")
    print(f"Number of training nodes: {data.train_mask.sum().item()}")
    print(f"Number of validation nodes: {data.val_mask.sum().item()}")
    print(f"Number of test nodes: {data.test_mask.sum().item()}")
    print(f"Zero feature vectors: {(data.x.sum(dim=1) == 0).sum().item()}")
    print("-------------------")
def print_node_indices(subgraph):
    """
    Print the indices of training, validation, and test nodes in the given subgraph.

    Args:
        subgraph: A graph or subgraph object that contains train_mask, val_mask, and test_mask attributes.
    """
    print(f"Training nodes in the subgraph: {subgraph.train_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of training nodes: {subgraph.train_mask.sum().item()}")
    print(f"Validation nodes in the subgraph: {subgraph.val_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of validation nodes: {subgraph.val_mask.sum().item()}")
    print(f"Test nodes in the subgraph: {subgraph.test_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of test nodes: {subgraph.test_mask.sum().item()}")
# Example usage:
# print_subgraph_stats(subgraph, "Original subgraph")
# print_subgraph_stats(expanded_subgraph, "Expanded subgraph")

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
from run import load_and_split_with_khop
data, dataset, clients_data, test_data,  = load_and_split_with_khop("Cora", device, num_clients=10, beta=0.5, hop=1)
# review data for client_0
sg = clients_data[0]
print_subgraph_stats(sg, "Client 0")
print_node_indices(sg)


Client 0 stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 7
Number of training nodes: 23
Number of validation nodes: 67
Number of test nodes: 120
Zero feature vectors: 241
-------------------
Training nodes in the subgraph: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
Number of training nodes: 23
Validation nodes in the subgraph: [23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
Number of validation nodes: 67
Test nodes in the subgraph: [224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 2

In [10]:
# load with fp and do the same thing
from run import load_and_split_with_feature_prop
data, dataset, clients_data, test_data,  = load_and_split_with_feature_prop("Cora", device, num_clients=10, beta=0.5, hop=1)
# review data for client_0
sg = clients_data[0]
print_subgraph_stats(sg, "Client 0")
print_node_indices(sg)


Client 0 stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 7
Number of training nodes: 23
Number of validation nodes: 67
Number of test nodes: 120
Zero feature vectors: 0
-------------------
Training nodes in the subgraph: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
Number of training nodes: 23
Validation nodes in the subgraph: [23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
Number of validation nodes: 67
Test nodes in the subgraph: [224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275

In [4]:
# examien a singraph from the kop thing
sg = clients_data[0]
# check number of nodes in the subgraph
print(f"Number of nodes in the subgraph: {sg.num_nodes}")
# check number of edges in the subgraph
print(f"Number of edges in the subgraph: {sg.edge_index.shape[1]}")
# check features of the subgraph
print(f"Features of the subgraph: {sg.x.shape}")
# check number of training nodes
print(f"Number of training nodes: {sg.train_mask.sum()}")
# check number of validation nodes
print(f"Number of validation nodes: {sg.val_mask.sum()}")
# check number of test nodes
print(f"Number of test nodes: {sg.test_mask.sum()}")
# number of nodes with zero feature vectors sum for each node then fidn the ones whose sum is zero
zero_features = (sg.x.sum(dim=1) == 0).sum()
print(f"Number of nodes with zero feature vectors: {zero_features}")













Number of nodes in the subgraph: 344
Number of edges in the subgraph: 1056
Features of the subgraph: torch.Size([344, 1433])
Number of training nodes: 23
Number of validation nodes: 67
Number of test nodes: 120
Number of nodes with zero feature vectors: 241


In [5]:
import torch

print(f"Number of GPUs available: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU available")

Number of GPUs available: 0
No GPU available


In [ ]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment
import os
import ray
ray.shutdown()

if __name__ == "__main__":
    # Define all options in a list
    # data_loading_options = ["split_dataset", "split_dataset_with_khop", "split_dataset_with_feature_prop"]
    data_loading_options = ["split_dataset_with_khop", "split_dataset_with_feature_prop", "split_dataset"]
    model_types = ["GCN"]
        
    # Load configuration once since it's common for all runs
    clients_num, beta, cfg = load_configuration()

    clients_num = 10

    # Main results directory
    main_dir = 'results'
    sub_dir = 'More_epochs_results3'
    # main_dir = 'New_results'
    
    dataset_name = "Cora"  # You can change this if you have different datasets

    # Loop over all combinations of data_loading_option and model_type
    for data_loading_option in data_loading_options:
        for model_type in model_types:
            # Create a structured directory for each experiment
            result_name = f"{dataset_name}_{data_loading_option}_{model_type}"
            results_dir = os.path.join(main_dir, sub_dir, result_name)

            # Create the directory if it doesn't exist
            os.makedirs(results_dir, exist_ok=True)

            print(f"Running experiment with data_loading_option: {data_loading_option}, model_type: {model_type}")
            result = main_experiment(clients_num, beta, data_loading_option, model_type, cfg, dataset_name=dataset_name, hop=2)
            
            # Replace literal '\n' with actual newline characters if necessary
            result_with_newlines = result.replace('\\n', '\n')
            
            # Create a unique filename for each combination
            filename = f'results_{dataset_name}_{data_loading_option}_{model_type}.txt'
            filepath = os.path.join(results_dir, filename)
            
            # Write the result to the text file
            with open(filepath, 'w') as f:
                f.write(result_with_newlines)
            
            print(f"Results saved to {filepath}\n")

Running experiment with data_loading_option: split_dataset_with_khop, model_type: GCN
DEVICE: cpu


2025-02-22 16:26:42,471	INFO worker.py:1770 -- Started a local Ray instance.


data_loading_option: split_dataset_with_khop
Data loaded
Device is cpu
Cora()
10


(FLClient pid=23882) 2025-02-22 16:26:51,073 - INFO - Epoch   0| Train Loss: 1.949| Train Accuracy: 0.145
(FLClient pid=23882) 2025-02-22 16:26:51,101 - INFO - Epoch   1| Train Loss: 1.937| Train Accuracy: 0.164
(FLClient pid=23882) 2025-02-22 16:26:51,110 - INFO - Epoch   2| Train Loss: 1.925| Train Accuracy: 0.200
(FLClient pid=23882) 2025-02-22 16:26:51,118 - INFO - Epoch   3| Train Loss: 1.911| Train Accuracy: 0.182
(FLClient pid=23882) 2025-02-22 16:26:51,130 - INFO - Epoch   4| Validation Loss: 1.946, Validation Accuracy: 0.145


Training done
Round 1
Train Loss: 1.960, Train Accuracy: 0.276
Train Loss: 1.946, Train Accuracy: 0.182
Train Loss: 1.959, Train Accuracy: 0.302
Train Loss: 1.912, Train Accuracy: 0.271
Train Loss: 1.916, Train Accuracy: 0.218
Train Loss: 1.958, Train Accuracy: 0.283
Train Loss: 1.970, Train Accuracy: 0.197
Train Loss: 1.986, Train Accuracy: 0.264
Train Loss: 1.922, Train Accuracy: 0.382
Train Loss: 1.925, Train Accuracy: 0.438
Round 2
Train Loss: 1.950, Train Accuracy: 0.362
Train Loss: 1.940, Train Accuracy: 0.291
Train Loss: 1.959, Train Accuracy: 0.206
Train Loss: 1.907, Train Accuracy: 0.354
Train Loss: 1.905, Train Accuracy: 0.273
Train Loss: 1.954, Train Accuracy: 0.300
Train Loss: 1.970, Train Accuracy: 0.197
Train Loss: 1.985, Train Accuracy: 0.245
Train Loss: 1.917, Train Accuracy: 0.338
Train Loss: 1.920, Train Accuracy: 0.583
Round 3
Train Loss: 1.940, Train Accuracy: 0.517
Train Loss: 1.931, Train Accuracy: 0.364
Train Loss: 1.954, Train Accuracy: 0.286
Train Loss: 1.897, 

...

data_loading_option: split_dataset
Data loaded
Device is cpu
Cora()
10


(FLClient pid=24425) 2025-02-22 16:32:09,551 - INFO - Epoch   0| Train Loss: 1.963| Train Accuracy: 0.143
(FLClient pid=24425) 2025-02-22 16:32:09,570 - INFO - Epoch   1| Train Loss: 1.683| Train Accuracy: 0.714
(FLClient pid=24425) 2025-02-22 16:32:09,578 - INFO - Epoch   2| Train Loss: 1.463| Train Accuracy: 0.786
(FLClient pid=24425) 2025-02-22 16:32:09,585 - INFO - Epoch   3| Train Loss: 1.238| Train Accuracy: 0.786
(FLClient pid=24425) 2025-02-22 16:32:09,591 - INFO - Epoch   4| Train Loss: 1.028| Train Accuracy: 0.857
(FLClient pid=24425) 2025-02-22 16:32:09,594 - INFO - Epoch   4| Validation Loss: 1.975, Validation Accuracy: 0.082
(FLClient pid=24425) 2025-02-22 16:32:14,640 - INFO - Epoch   3| Train Loss: 0.028| Train Accuracy: 1.000 [repeated 1149x across cluster]
(FLClient pid=24434) 2025-02-22 16:32:14,543 - INFO - Epoch   4| Validation Loss: 1.270, Validation Accuracy: 0.593 [repeated 229x across cluster]


Training done
Round 1
Train Loss: 1.975, Train Accuracy: 0.857
Train Loss: 2.011, Train Accuracy: 0.812
Train Loss: 2.030, Train Accuracy: 0.933
Train Loss: 1.755, Train Accuracy: 1.000
Train Loss: 1.811, Train Accuracy: 0.857
Train Loss: 1.651, Train Accuracy: 0.867
Train Loss: 1.900, Train Accuracy: 0.933
Train Loss: 1.893, Train Accuracy: 0.929
Train Loss: 1.870, Train Accuracy: 1.000
Train Loss: 1.849, Train Accuracy: 0.846
Round 2
Train Loss: 1.867, Train Accuracy: 0.929
Train Loss: 1.936, Train Accuracy: 0.938
Train Loss: 1.933, Train Accuracy: 1.000
Train Loss: 1.681, Train Accuracy: 1.000
Train Loss: 1.681, Train Accuracy: 1.000
Train Loss: 1.556, Train Accuracy: 1.000
Train Loss: 1.785, Train Accuracy: 1.000
Train Loss: 1.826, Train Accuracy: 0.929
Train Loss: 1.788, Train Accuracy: 1.000
Train Loss: 1.760, Train Accuracy: 1.000
Round 3
Train Loss: 1.759, Train Accuracy: 0.929
Train Loss: 1.864, Train Accuracy: 1.000
Train Loss: 1.838, Train Accuracy: 1.000
Train Loss: 1.592, 

(FLClient pid=24434) 2025-02-22 16:32:15,415 - INFO - Epoch   4| Train Loss: 0.017| Train Accuracy: 1.000 [repeated 346x across cluster]
(FLClient pid=24434) 2025-02-22 16:32:15,418 - INFO - Epoch   4| Validation Loss: 1.266, Validation Accuracy: 0.610 [repeated 70x across cluster]


Round 5 is complete
The global test results: [0.773, 0.772, 0.783, 0.782, 0.774]
The client test results: [0.6152659810138221, 0.6206624596735615, 0.6273366068212981, 0.6261974757119071, 0.6375210480769056]
The average global test results: 0.7767999999999999
The average client test results: 0.6253967142594988
The standad deviation global is: 0.004707440918375932
The standad deviation client is: 0.007435708374464774
Results saved to results/More_epochs_results2/Cora_split_dataset_GCN/results_Cora_split_dataset_GCN.txt



### Tests